In [1]:
import polars as pl
import yaml

with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [2]:
DIR = f"{config['dirs']['output']}/{config['city']}_ids_not_replaced"

In [3]:
for i in range(1):
    temp = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")

In [5]:
temp

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
"""0000051fc26276495c5a9ab1753a70…",1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,11
"""0000051fc26276495c5a9ab1753a70…",2,2024-10-02 10:23:47 CEST,2024-10-02 10:33:00 CEST,9m 13s,47.661216,19.076262,4.0,1.0,2,11
"""0000051fc26276495c5a9ab1753a70…",3,2024-10-03 10:50:40 CEST,2024-10-03 10:58:29 CEST,7m 49s,47.498402,19.0404,0.0,0.0,2,12
"""0000051fc26276495c5a9ab1753a70…",4,2024-10-04 12:41:20 CEST,2024-10-04 12:49:26 CEST,8m 6s,47.498402,19.0404,0.0,0.0,2,12
"""0000051fc26276495c5a9ab1753a70…",5,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,12
…,…,…,…,…,…,…,…,…,…,…
"""00ffd69f0786f191b711f219fd6605…",381,2024-11-17 18:27:21 CET,2024-11-17 18:37:33 CET,10m 12s,47.606706,19.270747,14.0,12.0,3,4
"""00ffd69f0786f191b711f219fd6605…",382,2024-11-17 21:11:36 CET,2024-11-17 21:26:20 CET,14m 44s,47.606829,19.270937,11.0,6.0,3,4
"""00ffd69f0786f191b711f219fd6605…",383,2024-11-18 04:47:22 CET,2024-11-18 05:13:19 CET,25m 57s,47.60688,19.270975,9.0,2.0,4,4


In [7]:
import polars_distance as pld

In [ ]:
data.select(
    pld.col("longitude_mean").dist.haversine("latitude_mean", "km").alias("haversine")
)

In [ ]:
data = temp.filter(pl.col.device_id.str.starts_with("0000051fc26276495c5a9ab1753a70"))[
    :3
].clone()

In [24]:
d = data.with_columns(
    pl.struct("longitude_mean", "latitude_mean")
    .alias("stop")
    .struct.rename_fields(["longitude", "latitude"]),
    pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean())
    .alias("centroid")
    .struct.rename_fields(["longitude", "latitude"]),
)
d

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,stop,centroid
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,struct[2],struct[2]
"""0000051fc26276495c5a9ab1753a70…",1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,11,"{19.076356,47.661166}","{19.064339,47.606928}"
"""0000051fc26276495c5a9ab1753a70…",2,2024-10-02 10:23:47 CEST,2024-10-02 10:33:00 CEST,9m 13s,47.661216,19.076262,4.0,1.0,2,11,"{19.076262,47.661216}","{19.064339,47.606928}"
"""0000051fc26276495c5a9ab1753a70…",3,2024-10-03 10:50:40 CEST,2024-10-03 10:58:29 CEST,7m 49s,47.498402,19.0404,0.0,0.0,2,12,"{19.0404,47.498402}","{19.064339,47.606928}"


In [25]:
d.with_columns(pld.col("centroid").dist.haversine("stop", "km").alias("haversine"))

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,stop,centroid,haversine
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,struct[2],struct[2],f64
"""0000051fc26276495c5a9ab1753a70…",1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,11,"{19.076356,47.661166}","{19.064339,47.606928}",6.097816
"""0000051fc26276495c5a9ab1753a70…",2,2024-10-02 10:23:47 CEST,2024-10-02 10:33:00 CEST,9m 13s,47.661216,19.076262,4.0,1.0,2,11,"{19.076262,47.661216}","{19.064339,47.606928}",6.102335
"""0000051fc26276495c5a9ab1753a70…",3,2024-10-03 10:50:40 CEST,2024-10-03 10:58:29 CEST,7m 49s,47.498402,19.0404,0.0,0.0,2,12,"{19.0404,47.498402}","{19.064339,47.606928}",12.200559


In [ ]:
def rg(data):
    d = data.with_columns(
        pl.struct("longitude_mean", "latitude_mean").alias("stop"),
        pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean()).alias(
            "centroid"
        ),
    )
    d.with_columns(pld.col("centroid").dist.haversine("stop", "km").alias("haversine"))
    rcm_lon = data["longitude_mean"].mean()
    rcm_lat = data["latitude_mean"].mean()
    n = data["interval_lenght"].dt.seconds
    data["diff_lon"] = (data["longitude_mean"] - rcm_lon) ** 2
    data["diff_lat"] = (data["latitude_mean"] - rcm_lat) ** 2
    freq = data.groupby(["x", "y"])["t"].count().to_dict()

In [ ]:
temp.group_by(pl.col.device_id).with_columns(pl.col.longitude_mean)

In [ ]:
def calculate_radius_of_gyration(data, weighted: bool = False):
    rcm_lon = data["lon"].mean()
    rcm_lat = data["lat"].mean()
    freq = data.groupby(["x", "y"])["t"].count().to_dict()
    ds = []
    w = 0
    for row in data.itertuples():
        w += freq[(row.x, row.y)]
        d = haversine((rcm_lat, rcm_lon), (row.lat, row.lon), unit=Unit.METERS) ** 2
        if weighted:
            d *= freq[(row.x, row.y)]
        ds.append(d)

    sum_ = np.sum(ds)
    N = len(data)
    if weighted:
        N = w
    return np.round(np.sqrt(sum_ / N) / 1000, 3)